<a href="https://colab.research.google.com/github/garvcodes/torchleet/blob/main/torch/easy/benchmark/bench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/torch/easy/benchmark/bench.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("benchmark", ...)`


## Problem: Add a Benchmark to Your PyTorch Code

### Problem Statement
You are tasked with implementing a simple neural network model with fully connected layers and adding benchmarking functionality to measure and display the time taken for each epoch of training and testing. The goal is to evaluate the model's performance and record the time taken for both training and testing phases.

### Requirements
1. **Define a Neural Network Model**:
   - Implement a simple feedforward neural network using fully connected layers (`nn.Linear`).
   - The network should be suitable for classification tasks.

2. **Benchmark Training and Testing**:
   - Measure the time taken for each epoch during training and display the elapsed time.
   - Measure and display the time taken for the testing phase after each epoch.

### Constraints
- The model should have at least two hidden layers with ReLU activations.
- Use the appropriate loss function and optimizer for training the model.
- Ensure that the benchmarking measures both the training and testing time accurately.

<details>
  <summary>💡 Hint</summary>
  Define the SimpleNN class:
  <br>
  Add two fully connected layers:
  <br>
  Apply a ReLU activation function to the first layer.
  <br>
  <br>
  Benchmark the Code:
  <br>
  Measure and print training time for each epoch.
  <br>
  Measure and print testing time along with accuracy.
</details>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time

In [2]:

# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.64MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 160kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.51MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.3MB/s]


In [9]:
print(type(train_dataset))
print(len(train_dataset))
print([
    name for name in dir(train_dataset)
    if not name.startswith("_")
])

image, label = train_dataset[0]
print(image.shape)
print(label)
print(image.dtype)
image = image.flatten()
print(image.shape)

<class 'torchvision.datasets.mnist.MNIST'>
60000
['class_to_idx', 'classes', 'data', 'download', 'extra_repr', 'mirrors', 'processed_folder', 'raw_folder', 'resources', 'root', 'target_transform', 'targets', 'test_data', 'test_file', 'test_labels', 'train', 'train_data', 'train_labels', 'training_file', 'transform', 'transforms']
torch.Size([1, 28, 28])
5
torch.float32
torch.Size([784])


In [18]:
# Define a simple neural network model
# TODO: Add layers to the model
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(
            in_features = 784,
            out_features=1400
        )

        self.fc2 = nn.Linear(
            in_features = 1400,
            out_features=10
        )

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        post_fc1 = torch.relu(self.fc1(x))
        return self.fc2(post_fc1)

# Initialize the model, loss function, and optimizer
model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [19]:
# Training loop with benchmarking
epochs = 5
for epoch in range(epochs):
    start_time = time.time()  # Start time for training
    for images, labels in train_loader:
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    end_time = time.time()  # End time for training
    training_time = end_time - start_time
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}, Time: {training_time:.4f}s")

Epoch [1/5], Loss: 0.5091, Time: 18.2287s
Epoch [2/5], Loss: 0.2349, Time: 17.5894s
Epoch [3/5], Loss: 0.4336, Time: 20.6439s
Epoch [4/5], Loss: 0.2888, Time: 18.8074s
Epoch [5/5], Loss: 0.3387, Time: 19.2436s


In [20]:
# Evaluate the model on the test set and benchmark the accuracy
correct = 0
total = 0
start_time = time.time()  # Start time for testing
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

end_time = time.time()  # End time for testing
testing_time = end_time - start_time
accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%, Testing Time: {testing_time:.4f}s")

Test Accuracy: 91.63%, Testing Time: 2.3681s
